In [0]:
-- each complete run in silver is one snapshot, numbered in time order
create or replace temp view snaps as
select observed_at, row_number() over (order by observed_at) snap_no, count(*) over () total_snaps
from (select distinct observed_at from cloud_pricing.silver.vm_prices)

In [0]:
create or replace temp view obs as
select v.*, s.snap_no, s.total_snaps,
lag(s.snap_no) over (partition by v.price_key order by s.snap_no) prev_snap,
lag(v.retail_price) over (partition by v.price_key order by s.snap_no) prev_price,
lag(v.effective_start_date) over (partition by v.price_key order by s.snap_no) prev_effective_date,
lead(s.snap_no) over (partition by v.price_key order by s.snap_no) next_snap
from cloud_pricing.silver.vm_prices v
join snaps s on v.observed_at = s.observed_at

In [0]:
-- new version when the key is seen first time, comes back after a gap, or the price moves
create or replace temp view versioned as
select *,
sum(case when prev_snap is null or prev_snap <> snap_no - 1 or prev_price <> retail_price then 1 else 0 end)
over (partition by price_key order by snap_no rows unbounded preceding) version_no
from obs

In [0]:
-- full rebuild from silver every run. data is small and this keeps reruns simple
create or replace table cloud_pricing.gold.price_history as
select h.price_key, h.version_no, h.region, h.arm_sku_name, h.meter_name, h.product_name, h.os, h.capacity_type,
h.price_type, h.term, h.unit_of_measure, h.currency_code, h.retail_price, h.effective_start_date,
h.valid_from, s.observed_at valid_to, h.last_seen_at, h.last_snap = h.total_snaps is_current
from (
select price_key, version_no,
max_by(region, snap_no) region, max_by(arm_sku_name, snap_no) arm_sku_name, max_by(meter_name, snap_no) meter_name,
max_by(product_name, snap_no) product_name, max_by(os, snap_no) os, max_by(capacity_type, snap_no) capacity_type,
max_by(price_type, snap_no) price_type, max_by(term, snap_no) term, max_by(unit_of_measure, snap_no) unit_of_measure,
max_by(currency_code, snap_no) currency_code, max_by(retail_price, snap_no) retail_price,
max_by(effective_start_date, snap_no) effective_start_date,
min(observed_at) valid_from, max(observed_at) last_seen_at, max(snap_no) last_snap, max(total_snaps) total_snaps
from versioned
group by price_key, version_no
) h
left join snaps s on s.snap_no = h.last_snap + 1

In [0]:
create or replace table cloud_pricing.gold.price_changes as
select e.price_key, e.change_type, s.observed_at detected_at, e.old_price, e.new_price,
case when e.change_type in ('increase', 'decrease') then round((e.new_price - e.old_price) / e.old_price * 100, 2) end pct_change,
e.old_effective_date, e.new_effective_date,
e.region, e.arm_sku_name, e.meter_name, e.os, e.capacity_type, e.price_type, e.term
from (
select price_key, snap_no event_snap,
case when retail_price > prev_price then 'increase' else 'decrease' end change_type,
prev_price old_price, retail_price new_price, prev_effective_date old_effective_date, effective_start_date new_effective_date,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where prev_snap = snap_no - 1 and retail_price <> prev_price
union all
-- first snapshot is the baseline, nothing in it counts as new
select price_key, snap_no, 'new', null, retail_price, null, effective_start_date,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where prev_snap is null and snap_no > 1
union all
select price_key, snap_no, 'reappeared', prev_price, retail_price, prev_effective_date, effective_start_date,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where prev_snap < snap_no - 1
union all
-- not in the next complete snapshot, could still come back
select price_key, snap_no + 1, 'missing', retail_price, null, effective_start_date, null,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where snap_no < total_snaps and (next_snap is null or next_snap > snap_no + 1)
union all
-- only call it removed after two snapshots in a row without it
select price_key, snap_no + 2, 'removed', retail_price, null, effective_start_date, null,
region, arm_sku_name, meter_name, os, capacity_type, price_type, term
from obs where snap_no + 2 <= total_snaps and (next_snap is null or next_snap > snap_no + 2)
) e
join snaps s on s.snap_no = e.event_snap

In [0]:
-- current rows in history should match the latest silver snapshot exactly
select
(select count(*) from cloud_pricing.gold.price_history where is_current) current_rows,
(select count(*) from cloud_pricing.silver.vm_prices where observed_at = (select max(observed_at) from cloud_pricing.silver.vm_prices)) latest_snapshot_rows

In [0]:
select detected_at, change_type, region, count(*) changes
from cloud_pricing.gold.price_changes
group by detected_at, change_type, region
order by detected_at, change_type, region